In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ydata_profiling

from eda_utils.csv_loader import read_csv
from eda_utils.globals import *
from eda_utils.plotter import plot_partial_result, plot_title_page
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# IN_ROOT_DIR = pathlib.Path("ns-data-sources/spreading_potentials/multi_layer_networks/small_real").glob("*.csv")
# ANALYSED_NETWORK = 'toy_network'
# ANALYSED_NETWORK = 'lazega'
# ANALYSED_NETWORK = 'eu_transport_klm'
# ANALYSED_NETWORK = 'ckm_physicians'
# ANALYSED_NETWORK = 'eu_transportation'
# ANALYSED_NETWORK = 'aucs'

IN_ROOT_DIR = pathlib.Path("ns-data-sources/spreading_potentials/multi_layer_networks/arxiv_netscience_coauthorship").glob("**/*.csv")
ANALYSED_NETWORK = "arxiv_netscience_coauthorship"
# ANALYSED_NETWORK = "arxiv_netscience_coauthorship_math.oc"

OUT_ROOT_DIR = "eda"

In [ ]:
raw_csvs = []
for idx, file_path in enumerate(IN_ROOT_DIR):
    print(f"processing {idx}th file: {file_path.name}")
    raw_csvs.append(read_csv(file_path))

raw_csv = pd.concat(raw_csvs, axis=0, ignore_index=True)
print(f"\n\navailable networks: {raw_csv['network'].unique()}\n\n")
raw_csv.head()

In [ ]:
result_grouped = raw_csv.loc[raw_csv[NETWORK] == ANALYSED_NETWORK].groupby(
    by=[NETWORK, PROTOCOL, P, ACTOR]
)
result_mean = result_grouped.mean()
result_std = result_grouped.std()

result_mean.head()

In [ ]:
out_dir = pathlib.Path(f"{OUT_ROOT_DIR}/{ANALYSED_NETWORK}")
out_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
ydata_profiling.ProfileReport(result_mean, title=f"{ANALYSED_NETWORK} mean").to_file(f"{out_dir}/mean.html")
ydata_profiling.ProfileReport(result_std, title=f"{ANALYSED_NETWORK} std").to_file(f"{out_dir}/std.html")

In [ ]:
result_all = pd.merge(result_mean, result_std, left_index=True, right_index=True, suffixes=("_avg", "_std")).sort_index().reset_index(ACTOR)
result_all.head()

In [ ]:
simulated_cases = {idx_name: list(idx_lvl) for idx_name, idx_lvl in zip(result_all.index.names, result_all.index.levels)}

pdf = PdfPages(out_dir / (f"plots.pdf"))

for network in simulated_cases[NETWORK]:

    # title page - network
    partial_fig = plot_title_page(network)
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    # obtain y axis values' range for this network
    maxx = result_all.loc[network].max()
    ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
    sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
    pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
    pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

    # mean result across the network
    partial_fig = plot_partial_result(
        partial_result=result_all.loc[network].groupby(ACTOR).mean().reset_index(),
        suptitle=r"$\bf{" + f"net-{network}" + "}$",
        sl_range=sl_range,
        ex_range=ex_range,
        pi_range=pi_range,
        pt_range=pt_range,
    )
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    for proto in simulated_cases[PROTOCOL]:

        # title page - protocol
        partial_fig = plot_title_page(proto)
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # obtain y axis values' range for this network and protocol
        maxx = result_all.loc[network].loc[proto].max()
        ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
        sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
        pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
        pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

        # mean result across the network and protocol
        partial_fig = plot_partial_result(
            partial_result=result_all.loc[network].loc[proto].groupby(ACTOR).mean().reset_index(),
            suptitle=f"net-{network}--" + r"$\bf{" + f"proto-{proto}" + "}$",
            sl_range=sl_range,
            ex_range=ex_range,
            pi_range=pi_range,
            pt_range=pt_range,
        )
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # plot result for all particular experiments
        for p in simulated_cases[P]:
            partial_fig = plot_partial_result(
                partial_result=result_all.loc[network].loc[proto].loc[p],
                suptitle=f"net-{network}--proto-{proto}--" + r"$\bf{" + f"p-{p}" + "}$",
                sl_range=sl_range,
                ex_range=ex_range,
                pi_range=pi_range,
                pt_range=pt_range,
            )
            partial_fig.savefig(pdf, format="pdf")
            plt.close(partial_fig)

pdf.close()